# Phase 3 — Step 4: The Hierarchical Retrieval Agent

## Objective

This is the climax of the architecture. We compose every component built in Phase 2 and Phase 3 into a single end-to-end pipeline that goes **query → answer** while preserving Zero-Trust guarantees and avoiding the latency ceiling of a full-LLM router.

## Three-phase flow

| Phase | Component | What it does | Source |
|---|---|---|---|
| **A — Vector Route** | KSP Router-Index | Embeds the query, does an HNSW lookup against the 23 document KSPs, returns the top-2 documents and their home departments | Phase 3 Step 2 |
| **B — Targeted Asymmetric Retrieval** | Asymmetric Hybrid Fusion + Dynamic RRF + Parent-Child Reconstruction | Runs the ChromaDB + BM25 ensemble only inside the user's readable intersection of the routed departments, applies the Zero-Trust `where` filter on clearance + departments, then restores parent context for isolated PII fragments | Phase 2 Steps 2, 3, 4 |
| **C — Secure Generation** | Llama 3.2 via Ollama | Receives only the surviving, RBAC-validated chunks and produces a grounded answer | Phase 2 Step 1 (of phase-2-plus) |

## Zero-Trust composition

The router is content-based and *not* user-aware — it proposes target departments based on the query, not on who is asking. Zero-Trust is enforced at the chunk level inside Phase B, where the `where` filter combines two cuts:

- **clearance_level ≤ user_cl** (severity cap)
- **allowed_departments ∈ (user readable set) ∩ (router target set ∪ {"all"})** (scope cap)

If the intersection is empty, the user is asking about content outside their permissions; the pipeline returns 0 chunks and the LLM produces an honest refusal.

## What we measure

Per query:
- Target departments chosen by the router
- Chunks retrieved pre-RBAC vs. chunks surviving the RBAC filter
- How many fragments triggered parent-child reconstruction
- Phase-A / Phase-B / Phase-C latency and total system latency
- The LLM's final answer

In [1]:
# Cell 1 — Imports + Ollama + embedding model + ChromaDB client
import json, os, re, time, statistics, textwrap
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import requests
import chromadb
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from rank_bm25 import BM25Okapi

BASE         = os.path.dirname(os.path.abspath("__file__"))
PH2_CHUNKS   = os.path.join(BASE, "..", "..", "data", "results", "notebook_results", "ph2", "ph2_plus_augmented_chunks.json")
STEP2_JSON   = os.path.join(BASE, "..", "..", "data", "results", "notebook_results", "ph3", "ph3_step2_ingestion_agent.json")
RESULTS_DIR  = os.path.join(BASE, "..", "..", "data", "results", "notebook_results", "ph3")
os.makedirs(RESULTS_DIR, exist_ok=True)

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL      = "llama3.2"

r = requests.get("http://localhost:11434/api/tags", timeout=5)
assert any(MODEL in m["name"] for m in r.json()["models"])
print(f"Ollama ready. Model: {MODEL}")

embed  = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={"device": "cpu"})
chroma = chromadb.Client()
print("Embedding model + ChromaDB ready.")

Ollama ready. Model: llama3.2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Embedding model + ChromaDB ready.


## Cell 2 — Load corpus, KSPs, and build the parent index

In [2]:
# Cell 2 — Load chunks + KSPs + assign physical home_department + parent index

with open(PH2_CHUNKS, "r", encoding="utf-8") as f:
    corpus = json.load(f)
with open(STEP2_JSON, "r", encoding="utf-8") as f:
    step2 = json.load(f)

# A document tagged `allowed_departments="all"` is readable by everyone, but it
# must still live physically in ONE real department's collection. The mapping
# below assigns a `home_department` to each of the 7 "all"-tagged documents,
# based on which department OWNS the document (not who can read it).
HOME_DEPT_FOR_ALL: dict[str, str] = {
    "Witty-QuickGuide-EN.pdf":         "sales",        # product manual shipped with the kit
    "hr_onboarding_guide.md":          "hr",           # HR-owned procedure
    "fin_expense_policy.md":           "finance",      # finance-owned policy
    "product_witty_timer_specs.md":    "sales",        # product spec
    "product_photocell_alignment.md":  "sales",        # product manual
    "product_release_notes_v4.txt":    "sales",        # product changelog
    "memo_it_security_reminder.txt":   "engineering",  # IT-owned awareness memo
}
REAL_DEPTS = {"hr", "finance", "engineering", "legal", "sales"}

def home_dept_of(source: str, allowed_dept: str) -> str:
    if allowed_dept in REAL_DEPTS:
        return allowed_dept
    if allowed_dept == "all":
        if source not in HOME_DEPT_FOR_ALL:
            raise ValueError(f"No home_dept mapping for 'all'-tagged doc: {source}")
        return HOME_DEPT_FOR_ALL[source]
    raise ValueError(f"Unknown allowed_departments value: {allowed_dept!r} for {source}")

all_chunks: list[Document] = []
for src, chunks in corpus.items():
    for ch in chunks:
        meta = ch["metadata"].copy()
        meta.setdefault("contains_PII", False)
        meta.setdefault("sensitivity_types", [])
        # Add the physical-home field at chunk level (inherits from doc).
        # `allowed_departments` (read permission) stays untouched — can still be "all".
        allowed = str(meta.get("allowed_departments", "all")).lower()
        meta["home_department"] = home_dept_of(src, allowed)
        all_chunks.append(Document(page_content=ch["page_content"], metadata=meta))

# Per-document info: keep both fields explicit
doc_info: dict[str, dict] = {}
for ch in all_chunks:
    src = ch.metadata["source_file"]
    if src not in doc_info:
        doc_info[src] = {
            "allowed_departments": str(ch.metadata.get("allowed_departments", "all")).lower(),
            "home_department":     ch.metadata["home_department"],
            "base_clearance":      int(ch.metadata.get("clearance_level", 0)),
            "doc_type":            str(ch.metadata.get("doc_type", "unknown")).lower(),
        }

# Physical partition by HOME department — there is NO 'all' physical collection.
dept_chunks: dict[str, list[Document]] = defaultdict(list)
for ch in all_chunks:
    dept_chunks[ch.metadata["home_department"]].append(ch)

# Parent index: per source file, concatenated prose (non-PII) text.
_parent_parts: dict[str, list[str]] = defaultdict(list)
for ch in all_chunks:
    if not ch.metadata.get("contains_PII"):
        _parent_parts[ch.metadata["source_file"]].append(ch.page_content)
parent_by_source: dict[str, str] = {s: "\n".join(parts) for s, parts in _parent_parts.items()}

ksps = step2["ksps"]
print(f"Corpus:           {len(all_chunks)} chunks from {len(corpus)} documents")
print(f"KSPs loaded:      {len(ksps)} documents")
print(f"Parent texts:     {len(parent_by_source)} documents have non-PII prose")

print(f"\nPhysical home-department partition (NO 'all' collection):")
print(f"  {'home_dept':<14}{'Chunks':>7}")
assert "all" not in dept_chunks, "Sanity check failed: an 'all' physical partition exists."
for dept, chs in sorted(dept_chunks.items(), key=lambda kv: -len(kv[1])):
    print(f"  {dept:<14}{len(chs):>7}")
print(f"  {'TOTAL':<14}{sum(len(v) for v in dept_chunks.values()):>7}")

n_all_perm = sum(1 for ch in all_chunks if str(ch.metadata.get("allowed_departments", "")).lower() == "all")
print(f"\n  Chunks with allowed_departments='all' (read-permission): {n_all_perm}")
print(f"  These chunks live physically inside their home_department collections,")
print(f"  not in a separate 'all' collection — matching production design.")

Corpus:           228 chunks from 23 documents
KSPs loaded:      23 documents
Parent texts:     23 documents have non-PII prose

Physical home-department partition (NO 'all' collection):
  home_dept      Chunks
  sales              54
  engineering        50
  hr                 48
  finance            44
  legal              32
  TOTAL             228

  Chunks with allowed_departments='all' (read-permission): 49
  These chunks live physically inside their home_department collections,
  not in a separate 'all' collection — matching production design.


## Cell 3 — Build the indexes: 1 Router-Index + N per-department chunk indexes

**Router-Index** (`router_idx`): 23 KSP embeddings — what Phase A queries.

**Per-department chunk indexes**: one ChromaDB collection + one BM25 index per unique `allowed_departments` value. Each chunk lives in exactly one of those collections (physical partition = the architectural routing target). No global `where` filter — routing is enforced by *which* collections Phase B opens, and RBAC is a chunk-level post-filter applied in Python after retrieval.

In [3]:
# Cell 3 — Router-Index + per-department chunk collections (ChromaDB + BM25)

# --- Router-Index (KSPs) ---
# The router-index metadata stores `home_department` (where the doc lives),
# not `allowed_departments` (who may read it). The router decides routing,
# not access control.
try: chroma.delete_collection("router_idx")
except Exception: pass
router_idx = chroma.create_collection(name="router_idx", metadata={"hnsw:space": "cosine"})

ksp_texts = [k["ksp"] for k in ksps]
ksp_embs  = embed.embed_documents(ksp_texts)
router_idx.add(
    documents=ksp_texts,
    embeddings=ksp_embs,
    ids=[f"d{i:02d}" for i in range(len(ksps))],
    metadatas=[{"source": k["source"],
                "home_department": doc_info[k["source"]]["home_department"]}
               for k in ksps],
)
print(f"Router-Index : {router_idx.count()} KSP vectors\n")

# --- Per-department chunk indexes (one per real dept, never 'all') ---
def sanitize(m: dict) -> dict:
    out = {}
    for k, v in m.items():
        if isinstance(v, (str, int, float, bool)):
            out[k] = v
        elif isinstance(v, list):
            out[k] = json.dumps(v)
    return out

def tokenize(t: str) -> list[str]: return re.findall(r"\w+", t.lower())

dept_collections: dict[str, "chromadb.Collection"] = {}
dept_bm25:        dict[str, BM25Okapi] = {}
dept_tok_corpus:  dict[str, list[list[str]]] = {}

print(f"Per-department indexes (physical homes — no 'all'):")
print(f"  {'Collection':<26}{'Chunks':>7}  {'Embed time':>12}")
print(f"  " + "-" * 50)
t_total = time.time()
for dept, chunks in dept_chunks.items():
    coll_name = f"chunks_{dept}"
    try: chroma.delete_collection(coll_name)
    except Exception: pass

    t0 = time.time()
    texts = [d.page_content for d in chunks]
    embs  = embed.embed_documents(texts)
    coll  = chroma.create_collection(name=coll_name, metadata={"hnsw:space": "cosine"})
    coll.add(
        documents=texts,
        embeddings=embs,
        ids=[f"{dept}_{i:04d}" for i in range(len(chunks))],
        metadatas=[sanitize(d.metadata) for d in chunks],
    )
    dept_collections[dept] = coll

    tok = [tokenize(t) for t in texts]
    dept_bm25[dept]       = BM25Okapi(tok)
    dept_tok_corpus[dept] = tok

    print(f"  {coll_name:<26}{len(chunks):>7}  {time.time()-t0:>11.1f}s")

print(f"  " + "-" * 50)
print(f"  {'TOTAL':<26}{sum(c.count() for c in dept_collections.values()):>7}  {time.time()-t_total:>11.1f}s")
print(f"\nCollections created: {sorted(dept_collections.keys())}  (count={len(dept_collections)})")
assert "all" not in dept_collections, "Sanity check failed: an 'all' physical collection was created."

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Router-Index : 23 KSP vectors

Per-department indexes (physical homes — no 'all'):
  Collection                 Chunks    Embed time
  --------------------------------------------------


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


...

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


  chunks_engineering             50          0.3s


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


  chunks_hr                      48          0.2s
  --------------------------------------------------
  TOTAL                         228          2.0s

Collections created: ['engineering', 'finance', 'hr', 'legal', 'sales']  (count=5)


## Cell 4 — Phase A: Vector Route + intent classifier

Two functions:

- `phase_a_route(query)` — embed + HNSW over 23 KSPs, take the top-2 documents, resolve their `department`, return the unique set.
- `pick_alpha(query)` — Phase 2 Step 2's Tier-Priority intent classifier that picks the RRF α (0.2 for exact/PII patterns, 0.8 for conceptual).

In [4]:
# Cell 4 — Phase A + intent classifier

TIER1_PATTERNS = [
    r"\b(?:password|contrase[ñn]a|pwd|api.?key|token)\b",
    r"\bIBAN\b", r"\bSWIFT\b", r"\bDNI\b",
]
TIER2_PATTERNS = [
    r"\bCLI-\d+\b", r"\bEMP-\d+\b", r"\bIR-\d+",
    r"\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b",
    r"\b(?:port|puerto)\s+\d+\b", r"\bUDP\b", r"\bTCP\b",
]

def pick_alpha(query: str) -> tuple[float, str]:
    for p in TIER1_PATTERNS:
        if re.search(p, query, re.IGNORECASE):
            return 0.2, "tier1_secret"
    for p in TIER2_PATTERNS:
        if re.search(p, query, re.IGNORECASE):
            return 0.2, "tier2_exact"
    return 0.8, "conceptual"

def phase_a_route(query: str, top_k: int = 2) -> dict:
    """Embed + HNSW over the KSPs → top-k docs and their HOME departments.
       The router never returns 'all' — every doc has a real home_department."""
    t0 = time.perf_counter()
    qe = embed.embed_query(query)
    r  = router_idx.query(query_embeddings=[qe], n_results=top_k,
                          include=["metadatas", "distances"])
    top_docs   = [r["metadatas"][0][i]["source"]          for i in range(top_k)]
    top_depts  = [r["metadatas"][0][i]["home_department"] for i in range(top_k)]
    top_dists  = [round(r["distances"][0][i], 4)          for i in range(top_k)]
    target_depts = sorted(set(top_depts))
    return {
        "top_docs":     top_docs,
        "top_depts":    top_depts,
        "top_dists":    top_dists,
        "target_depts": target_depts,
        "latency_s":    round(time.perf_counter() - t0, 4),
    }

# Smoke test
smoke = phase_a_route("What UDP port for the corporate VPN client?")
print(f"Phase A smoke: top docs = {smoke['top_docs']}")
print(f"               top home_depts = {smoke['top_depts']}")
print(f"               target_depts = {smoke['target_depts']}")
print(f"               latency = {smoke['latency_s']*1000:.1f} ms")
a, kind = pick_alpha("What UDP port for the corporate VPN client?")
print(f"Intent classifier: α={a} ({kind})")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Phase A smoke: top docs = ['it_vpn_config_guide.md', 'distribution-contract-2026.docx']
               top home_depts = ['engineering', 'legal']
               target_depts = ['engineering', 'legal']
               latency = 37.4 ms
Intent classifier: α=0.2 (tier2_exact)


## Cell 5 — Phase B: Targeted Asymmetric Hybrid Fusion + Parent-Child Reconstruction

Flow:

1. **Open ONLY the routed physical collections.** `target_depts` from Phase A is a subset of {hr, finance, engineering, legal, sales} — never `"all"`, since every document has a real `home_department`.
2. **ChromaDB branch** — HNSW query inside each routed collection.
3. **BM25 branch** — per-collection BM25 query inside each routed collection.
4. **Global merge-sort** before assigning RRF ranks (corrected RRF). When more than one collection is routed, the candidates from each are *pooled* and re-ranked by their raw score across the union before the RRF formula assigns `rank → score`. This avoids the artefact where a chunk's RRF rank reflects its position inside its own collection rather than its true relevance vs. the global candidate set.
5. **Dynamic RRF fusion** with α picked by the intent classifier (0.2 for exact/PII patterns, 0.8 for conceptual queries).
6. **RBAC post-filter, chunk by chunk, in Python**: keep iff `chunk.clearance_level ≤ user_cl` AND (`chunk.allowed_departments == user.dept` OR `chunk.allowed_departments == "all"`).
7. **Parent-child reconstruction** for short PII fragments that survive the filter — attach the parent prose from the same source file.

`pre_rbac` counts distinct chunks retrieved from the routed physical collections before the per-chunk RBAC gate; `post_rbac` counts what survived. Their difference is the Zero-Trust cost.

In [5]:
# Cell 5 — Phase B implementation (physical-collection routing + global merge-sort RRF + Python RBAC post-filter)

CHROMA_POOL_PER_COLL = 10
BM25_POOL_PER_COLL   = 30
FINAL_K              = 5
RK                   = 60   # RRF constant

def _chroma_search_coll(query: str, coll, pool: int) -> list[dict]:
    qe = embed.embed_query(query)
    k  = min(pool, coll.count())
    if k == 0:
        return []
    r = coll.query(query_embeddings=[qe], n_results=k,
                   include=["metadatas", "distances", "documents"])
    return [{"source": m["source_file"], "chunk_id": m.get("chunk_id", ""),
             "text": d, "distance": dist,
             "cl":   int(m.get("clearance_level", 0)),
             "dept": str(m.get("allowed_departments", "all")).lower()}
            for m, d, dist in zip(r["metadatas"][0], r["documents"][0], r["distances"][0])]

def _bm25_search_idx(query: str, bm25_idx: BM25Okapi, chunks: list[Document], pool: int) -> list[dict]:
    if not chunks:
        return []
    scores = bm25_idx.get_scores(tokenize(query))
    order  = np.argsort(scores)[::-1][:pool]
    out = []
    for i in order:
        if scores[i] <= 0:
            break
        d = chunks[i]
        out.append({"source": d.metadata["source_file"],
                    "chunk_id": d.metadata.get("chunk_id", ""),
                    "text": d.page_content, "score": float(scores[i]),
                    "cl":   int(d.metadata.get("clearance_level", 0)),
                    "dept": str(d.metadata.get("allowed_departments", "all")).lower()})
    return out

def _rrf_fuse(chr_r: list[dict], bm_r: list[dict], alpha: float, rk: int = RK) -> list[dict]:
    """Weighted RRF.  Caller must have pre-sorted both lists by global relevance
    (chr_r ascending by `distance`, bm_r descending by `score`) so that the
    `rank` derived from list position is meaningful across collections."""
    fused: dict[tuple, dict] = {}
    for rank, c in enumerate(chr_r, 1):
        key = (c["source"], c["chunk_id"])
        fused.setdefault(key, {**c, "rrf": 0.0})
        fused[key]["rrf"] += alpha / (rk + rank)
    for rank, c in enumerate(bm_r, 1):
        key = (c["source"], c["chunk_id"])
        fused.setdefault(key, {**c, "rrf": 0.0})
        fused[key]["rrf"] += (1 - alpha) / (rk + rank)
    return sorted(fused.values(), key=lambda x: -x["rrf"])

_PII_HINTS = re.compile(
    r"(?:IBAN|SWIFT|password|DNI:|EMP-\d|CLI-\d|\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})",
    re.IGNORECASE,
)

def _parent_child(ch: dict) -> dict:
    text = ch["text"]
    if len(text) < 120 and _PII_HINTS.search(text):
        parent = parent_by_source.get(ch["source"], "")
        if parent and len(parent) > 40:
            return {**ch, "text": f"[CONTEXT — parent paragraph from same document]\n{parent[:400]}\n[SENSITIVE FRAGMENT]\n{text}",
                    "parent_merged": True}
    return {**ch, "parent_merged": False}

def phase_b_retrieve(query: str, user_cl: int, user_dept: str, target_depts: list[str]) -> dict:
    t0 = time.perf_counter()
    alpha, intent = pick_alpha(query)

    # 1. Search inside the physically-routed collections (pool the candidates)
    chr_candidates: list[dict] = []
    bm_candidates:  list[dict] = []
    queried_colls: list[str]   = []
    for dept in target_depts:
        if dept not in dept_collections:
            # Should never trigger now: target_depts is always a subset of real depts.
            continue
        queried_colls.append(dept)
        chr_candidates.extend(_chroma_search_coll(query, dept_collections[dept], CHROMA_POOL_PER_COLL))
        bm_candidates.extend(_bm25_search_idx( query, dept_bm25[dept], dept_chunks[dept], BM25_POOL_PER_COLL))

    # 2. GLOBAL MERGE-SORT before assigning RRF ranks.
    #    ChromaDB: smaller cosine distance = better → ascending sort.
    #    BM25:     higher score = better         → descending sort.
    #    Without this step, a candidate's RRF rank would reflect its position
    #    inside its own collection block rather than its true relevance vs.
    #    the union of all routed collections.
    chr_candidates.sort(key=lambda c: c["distance"])
    bm_candidates.sort( key=lambda c: -c["score"])

    # 3. RRF fusion over the globally-ranked candidates
    fused     = _rrf_fuse(chr_candidates, bm_candidates, alpha)
    pre_rbac  = len(fused)

    # 4. RBAC post-filter, chunk by chunk
    def rbac_allows(c: dict) -> bool:
        return c["cl"] <= user_cl and (c["dept"] == user_dept or c["dept"] == "all")
    surviving = [c for c in fused if rbac_allows(c)]
    post_rbac = len(surviving)

    # 5. Parent-child reconstruction on the top-K survivors
    final = [_parent_child(c) for c in surviving[:FINAL_K]]

    access_denied = (pre_rbac > 0 and post_rbac == 0)

    return {
        "alpha": alpha, "intent": intent,
        "target_depts":        target_depts,
        "collections_queried": queried_colls,
        "pre_rbac":            pre_rbac,
        "post_rbac":           post_rbac,
        "rbac_dropped":        pre_rbac - post_rbac,
        "final_chunks":        final,
        "parent_merges":       sum(1 for c in final if c.get("parent_merged", False)),
        "access_denied":       access_denied,
        "latency_s":           round(time.perf_counter() - t0, 4),
    }

# Smoke test (same dept on both target and user → no RBAC drops expected)
sm = phase_b_retrieve("What UDP port for the corporate VPN client?",
                      user_cl=2, user_dept="engineering",
                      target_depts=["engineering"])
print(f"Phase B smoke: α={sm['alpha']} ({sm['intent']})")
print(f"               collections queried = {sm['collections_queried']}")
print(f"               pre-RBAC={sm['pre_rbac']}  post-RBAC={sm['post_rbac']}  dropped={sm['rbac_dropped']}")
print(f"               final chunks delivered = {len(sm['final_chunks'])}")
print(f"               latency = {sm['latency_s']*1000:.1f} ms")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Phase B smoke: α=0.2 (tier2_exact)
               collections queried = ['engineering']
               pre-RBAC=20  post-RBAC=20  dropped=0
               final chunks delivered = 5
               latency = 9.1 ms


## Cell 6 — Phase C: Secure Generation

In [6]:
# Cell 6 — Secure LLM generation

SYSTEM_PROMPT = (
    "You are a corporate knowledge assistant for an enterprise on-premise RAG system. "
    "Answer the user's question based EXCLUSIVELY on the provided context. "
    "Rules: (1) do not invent facts; (2) if the context is insufficient or empty, "
    "reply honestly that the information is not available to you; "
    "(3) keep the answer concise, 3-5 sentences."
)

def phase_c_generate(query: str, chunks: list[dict]) -> dict:
    t0 = time.perf_counter()
    if not chunks:
        ctx = "[NO ACCESSIBLE CONTEXT — the RBAC filter removed all candidate chunks.]"
    else:
        ctx = "\n---\n".join(c["text"] for c in chunks)
    prompt = f"{SYSTEM_PROMPT}\n\nCONTEXT:\n{ctx}\n\nQUESTION: {query}\n\nANSWER:"
    resp = requests.post(OLLAMA_URL, json={
        "model": MODEL, "prompt": prompt, "stream": False,
        "options": {"temperature": 0.1, "num_predict": 260, "num_ctx": 8192},
    }, timeout=180)
    ans = resp.json().get("response", "").strip()
    return {"answer": ans, "latency_s": round(time.perf_counter()-t0, 3),
            "context_chars": len(ctx)}

# Smoke test
sm_c = phase_c_generate("Say 'hello' in one word.", [])
print(f"Phase C smoke: \"{sm_c['answer']}\" ({sm_c['latency_s']}s)")

Phase C smoke: "I'm unable to provide an answer as there is no accessible context." (10.421s)


## Cell 7 — End-to-end pipeline + 5 test scenarios

| # | User | Dept | Clearance | Query | Expected outcome |
|---|---|---|---|---|---|
| Q1 | Luca (DevOps) | engineering | 2 | VPN UDP port | Answer from IT |
| Q2 | Chiara (CFO) | finance | 3 | Q1 2026 revenue by region | Answer from Finance |
| Q3 | Alessandra (HR Mgr) | hr | 2 | Travel reimbursement for new hires | Answer (ambiguous: finance ∪ hr) |
| Q4 | Valentina (Legal) | legal | 2 | Federation supply-contract warranty | Answer from Legal |
| Q5 | **Diego (Sales Intern)** | **sales** | **1** | IBAN of the Spanish distributor | **Refused** — cl=3 content + cross-dept boundary |

In [7]:
# Cell 7 — Scenarios + end-to-end execution with detailed logging

@dataclass
class Scenario:
    qid: str
    user_name: str
    user_dept: str
    user_cl: int
    query: str
    expected: str   # "answer" | "refused"

SCENARIOS = [
    Scenario("Q1", "Luca (DevOps Engineer)", "engineering", 2,
             "Which UDP port must the firewall allow for the corporate VPN client to connect?",
             "answer"),
    Scenario("Q2", "Chiara (CFO)", "finance", 3,
             "Summarise Q1 2026 revenue broken down by geographic region.",
             "answer"),
    Scenario("Q3", "Alessandra (HR Manager)", "hr", 2,
             "What is the process for reimbursing business travel expenses for a new hire?",
             "answer"),
    Scenario("Q4", "Valentina (Legal Counsel)", "legal", 2,
             "What is the warranty period offered in the Spanish Athletics Federation supply contract?",
             "answer"),
    Scenario("Q5", "Diego (Sales Intern)", "sales", 1,
             "Which IBAN does Microgate use to collect payments from the Spanish distributor?",
             "refused"),
]

def pipeline(sc: Scenario) -> dict:
    a = phase_a_route(sc.query, top_k=2)
    b = phase_b_retrieve(sc.query, sc.user_cl, sc.user_dept, a["target_depts"])
    c = phase_c_generate(sc.query, b["final_chunks"])
    total = round(a["latency_s"] + b["latency_s"] + c["latency_s"], 3)
    return {"scenario": sc, "A": a, "B": b, "C": c, "total_s": total}

results: list[dict] = []
print("=" * 100)
print("  HIERARCHICAL RETRIEVAL AGENT — 5 SCENARIO EXECUTION")
print("=" * 100)

for sc in SCENARIOS:
    res = pipeline(sc)
    results.append(res)
    a, b, c = res["A"], res["B"], res["C"]

    print(f"\n► {sc.qid}  user={sc.user_name}   dept={sc.user_dept}  cl={sc.user_cl}")
    print(f"    query: \"{sc.query}\"")

    print(f"\n    [PHASE A] Vector Route  ({a['latency_s']*1000:.1f} ms)")
    print(f"       Router top-2 documents:")
    for d, dep, dist in zip(a["top_docs"], a["top_depts"], a["top_dists"]):
        print(f"         • {d:<40}  dept={dep:<12}  cos_dist={dist}")
    print(f"       → target_depts (physical collections to open): {a['target_depts']}")

    print(f"\n    [PHASE B] Targeted Asymmetric Retrieval  ({b['latency_s']*1000:.1f} ms)")
    print(f"       Intent: α={b['alpha']} ({b['intent']})")
    print(f"       Collections opened: {b['collections_queried']}")
    print(f"       Chunks retrieved from those collections : {b['pre_rbac']}")
    print(f"       Chunks surviving RBAC post-filter        : {b['post_rbac']}   (dropped {b['rbac_dropped']} — cl or dept mismatch)")
    print(f"       Parent-child reconstructions             : {b['parent_merges']}")
    if b["final_chunks"]:
        print(f"       Top final chunks delivered to LLM:")
        for i, ch in enumerate(b["final_chunks"], 1):
            tag = " Ⓟ" if ch.get("parent_merged") else ""
            src = ch["source"][:34]
            preview = ch["text"][:80].replace("\n", " ")
            print(f"         #{i} {src:<36} cl={ch['cl']} dept={ch['dept']:<12}{tag}  \"{preview}...\"")
    elif b["access_denied"]:
        print(f"       ** ACCESS DENIED ** — RBAC rejected every chunk in the routed collections.")
    else:
        print(f"       (routed collections returned no candidates)")

    print(f"\n    [PHASE C] Secure Generation  ({c['latency_s']:.2f} s)")
    print(f"       Context length: {c['context_chars']} chars")
    print(f"       LLM answer:")
    for line in textwrap.wrap(c["answer"], width=95):
        print(f"         {line}")

    print(f"\n    ● TOTAL LATENCY = {res['total_s']:.2f}s   "
          f"(route {a['latency_s']*1000:.0f}ms + retrieve {b['latency_s']*1000:.0f}ms + generate {c['latency_s']*1000:.0f}ms)")
    print("-" * 100)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  HIERARCHICAL RETRIEVAL AGENT — 5 SCENARIO EXECUTION


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



► Q1  user=Luca (DevOps Engineer)   dept=engineering  cl=2
    query: "Which UDP port must the firewall allow for the corporate VPN client to connect?"

    [PHASE A] Vector Route  (14.4 ms)
       Router top-2 documents:
         • it_vpn_config_guide.md                    dept=engineering   cos_dist=0.4685
         • distribution-contract-2026.docx           dept=legal         cos_dist=0.8325
       → target_depts (physical collections to open): ['engineering', 'legal']

    [PHASE B] Targeted Asymmetric Retrieval  (23.4 ms)
       Intent: α=0.2 (tier2_exact)
       Collections opened: ['engineering', 'legal']
       Chunks retrieved from those collections : 36
       Chunks surviving RBAC post-filter        : 26   (dropped 10 — cl or dept mismatch)
       Parent-child reconstructions             : 0
       Top final chunks delivered to LLM:
         #1 it_vpn_config_guide.md               cl=2 dept=engineering   "All remote employees must connect via the corporate WireGuard VPN bef

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



► Q2  user=Chiara (CFO)   dept=finance  cl=3
    query: "Summarise Q1 2026 revenue broken down by geographic region."

    [PHASE A] Vector Route  (10.2 ms)
       Router top-2 documents:
         • fin_q1_2026_revenue.txt                   dept=finance       cos_dist=0.447
         • memo_q2_sales_targets.txt                 dept=sales         cos_dist=0.4647
       → target_depts (physical collections to open): ['finance', 'sales']

    [PHASE B] Targeted Asymmetric Retrieval  (20.6 ms)
       Intent: α=0.8 (conceptual)
       Collections opened: ['finance', 'sales']
       Chunks retrieved from those collections : 39
       Chunks surviving RBAC post-filter        : 27   (dropped 12 — cl or dept mismatch)
       Parent-child reconstructions             : 0
       Top final chunks delivered to LLM:
         #1 fin_q1_2026_revenue.txt              cl=3 dept=finance       "1. Quarterly Summary Total gross revenue for Q1 2026 reached €5.1 million, a 21%..."
         #2 fin_q1_2026_reve


► Q3  user=Alessandra (HR Manager)   dept=hr  cl=2
    query: "What is the process for reimbursing business travel expenses for a new hire?"

    [PHASE A] Vector Route  (11.0 ms)
       Router top-2 documents:
         • fin_expense_policy.md                     dept=finance       cos_dist=0.6282
         • hr_onboarding_guide.md                    dept=hr            cos_dist=0.856
       → target_depts (physical collections to open): ['finance', 'hr']

    [PHASE B] Targeted Asymmetric Retrieval  (20.2 ms)
       Intent: α=0.8 (conceptual)
       Collections opened: ['finance', 'hr']
       Chunks retrieved from those collections : 44
       Chunks surviving RBAC post-filter        : 14   (dropped 30 — cl or dept mismatch)
       Parent-child reconstructions             : 0
       Top final chunks delivered to LLM:
         #1 fin_expense_policy.md                cl=1 dept=all           "## 1. Travel Expenses All business travel must be pre-approved by the department..."
         #2


► Q4  user=Valentina (Legal Counsel)   dept=legal  cl=2
    query: "What is the warranty period offered in the Spanish Athletics Federation supply contract?"

    [PHASE A] Vector Route  (10.9 ms)
       Router top-2 documents:
         • contract_fitplus_maintenance.txt          dept=sales         cos_dist=0.5533
         • distribution-contract-2026.docx           dept=legal         cos_dist=0.5682
       → target_depts (physical collections to open): ['legal', 'sales']

    [PHASE B] Targeted Asymmetric Retrieval  (19.0 ms)
       Intent: α=0.8 (conceptual)
       Collections opened: ['legal', 'sales']
       Chunks retrieved from those collections : 32
       Chunks surviving RBAC post-filter        : 17   (dropped 15 — cl or dept mismatch)
       Parent-child reconstructions             : 0
       Top final chunks delivered to LLM:
         #1 contract_federacion_atletismo.txt    cl=2 dept=legal         "El precio unitario acordado es de €780..."
         #2 contract_federacion_a


► Q5  user=Diego (Sales Intern)   dept=sales  cl=1
    query: "Which IBAN does Microgate use to collect payments from the Spanish distributor?"

    [PHASE A] Vector Route  (10.3 ms)
       Router top-2 documents:
         • fin_expense_policy.md                     dept=finance       cos_dist=0.5544
         • hr_salary_bands_2026.txt                  dept=hr            cos_dist=0.6048
       → target_depts (physical collections to open): ['finance', 'hr']

    [PHASE B] Targeted Asymmetric Retrieval  (19.1 ms)
       Intent: α=0.2 (tier1_secret)
       Collections opened: ['finance', 'hr']
       Chunks retrieved from those collections : 62
       Chunks surviving RBAC post-filter        : 9   (dropped 53 — cl or dept mismatch)
       Parent-child reconstructions             : 0
       Top final chunks delivered to LLM:
         #1 hr_onboarding_guide.md               cl=0 dept=all           "## Welcome to Microgate! This guide will help you get started during your first ..."
      

## Cell 8 — Aggregate metrics + export

In [8]:
# Cell 8 — Aggregate stats + JSON export

rows = []
for res in results:
    sc, a, b, c = res["scenario"], res["A"], res["B"], res["C"]
    rows.append({
        "qid": sc.qid, "user": sc.user_name, "dept": sc.user_dept, "cl": sc.user_cl,
        "expected": sc.expected,
        "query": sc.query,
        "target_depts":        a["target_depts"],
        "collections_queried": b["collections_queried"],
        "alpha": b["alpha"], "intent": b["intent"],
        "pre_rbac": b["pre_rbac"], "post_rbac": b["post_rbac"],
        "rbac_dropped": b["rbac_dropped"],
        "parent_merges": b["parent_merges"],
        "access_denied": b["access_denied"],
        "lat_route_ms":    round(a["latency_s"]*1000, 1),
        "lat_retrieve_ms": round(b["latency_s"]*1000, 1),
        "lat_generate_s":  round(c["latency_s"], 2),
        "total_s":         res["total_s"],
        "answer":          c["answer"],
    })
df = pd.DataFrame(rows)

lat_route    = [r["lat_route_ms"]    for r in rows]
lat_retrieve = [r["lat_retrieve_ms"] for r in rows]
lat_generate = [r["lat_generate_s"]  for r in rows]
totals       = [r["total_s"]         for r in rows]

print("=" * 76)
print("  AGGREGATE LATENCY  (5 queries)")
print("=" * 76)
print(f"  {'Phase':<26}{'mean':>10}{'median':>10}{'min':>10}{'max':>10}")
print("  " + "-" * 66)
def stats(xs, fmt):
    return f"{fmt.format(statistics.mean(xs)):>10}{fmt.format(statistics.median(xs)):>10}{fmt.format(min(xs)):>10}{fmt.format(max(xs)):>10}"
print(f"  {'A Vector Route (ms)':<26}{stats(lat_route,    '{:.1f}')}")
print(f"  {'B Asym. Retrieval (ms)':<26}{stats(lat_retrieve, '{:.1f}')}")
print(f"  {'C Secure Generation (s)':<26}{stats(lat_generate, '{:.2f}')}")
print(f"  {'TOTAL (s)':<26}{stats(totals,      '{:.2f}')}")

print("\n" + "=" * 76)
print("  RBAC IMPACT")
print("=" * 76)
tot_pre  = sum(r["pre_rbac"]  for r in rows)
tot_post = sum(r["post_rbac"] for r in rows)
print(f"  Chunks retrieved from routed collections (pre-RBAC): {tot_pre}")
print(f"  Chunks surviving RBAC post-filter:                    {tot_post}")
print(f"  RBAC drop rate:                                        {(tot_pre-tot_post)/max(tot_pre,1)*100:.0f}%")
print(f"  Parent-child reconstructions performed:                {sum(r['parent_merges'] for r in rows)}")

denied = [r for r in rows if r["access_denied"]]
print(f"\n  Scenarios where access was denied: {len(denied)} / {len(rows)}")
for r in denied:
    print(f"    • {r['qid']} {r['user']} → {r['query'][:70]}")

# Export
export = {
    "step": "Phase 3 - Step 4: Hierarchical Retrieval Agent",
    "model": MODEL,
    "architecture": "physical per-department ChromaDB collections + per-department BM25 indexes, Python RBAC post-filter",
    "n_queries": len(rows),
    "latency_summary": {
        "phase_a_ms":    {"mean": round(statistics.mean(lat_route), 2),    "max": max(lat_route)},
        "phase_b_ms":    {"mean": round(statistics.mean(lat_retrieve), 2), "max": max(lat_retrieve)},
        "phase_c_s":     {"mean": round(statistics.mean(lat_generate), 2), "max": max(lat_generate)},
        "total_s":       {"mean": round(statistics.mean(totals), 2),       "max": max(totals)},
    },
    "rbac": {
        "total_pre_rbac":  tot_pre,
        "total_post_rbac": tot_post,
        "drop_pct":        round((tot_pre-tot_post)/max(tot_pre,1)*100, 1),
        "parent_merges":   sum(r["parent_merges"] for r in rows),
        "access_denied":   len(denied),
    },
    "scenarios": rows,
}
json_path = os.path.join(RESULTS_DIR, "ph3_step4_hierarchical_agent.json")
csv_path  = os.path.join(RESULTS_DIR, "ph3_step4_hierarchical_agent.csv")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(export, f, indent=2, ensure_ascii=False, default=str)
df.to_csv(csv_path, index=False)
print(f"\nExported: {json_path}")
print(f"Exported: {csv_path}")

print("\n" + "=" * 72)
print("  STEP 4 SUMMARY  (physical per-dept collections)")
print("=" * 72)
print(f"  Collections created   : {len(dept_collections)}  ({sorted(dept_collections.keys())})")
print(f"  Scenarios executed    : {len(rows)}")
print(f"  Mean Phase A (route)  : {statistics.mean(lat_route):.1f} ms")
print(f"  Mean Phase B (RBAC)   : {statistics.mean(lat_retrieve):.1f} ms")
print(f"  Mean Phase C (LLM gen): {statistics.mean(lat_generate):.2f} s")
print(f"  Mean TOTAL latency    : {statistics.mean(totals):.2f} s")
print(f"  RBAC drop rate        : {(tot_pre-tot_post)/max(tot_pre,1)*100:.0f}%  (pre {tot_pre} → post {tot_post})")
print(f"  Access-denied queries : {len(denied)} / {len(rows)}")
print("\n  Awaiting approval before Step 5 (End-to-End Trade-off Analysis).")

  AGGREGATE LATENCY  (5 queries)
  Phase                           mean    median       min       max
  ------------------------------------------------------------------
  A Vector Route (ms)             11.4      10.9      10.2      14.4
  B Asym. Retrieval (ms)          20.5      20.2      19.0      23.4
  C Secure Generation (s)         3.66      3.72      3.35      4.05
  TOTAL (s)                       3.69      3.75      3.38      4.08

  RBAC IMPACT
  Chunks retrieved from routed collections (pre-RBAC): 213
  Chunks surviving RBAC post-filter:                    93
  RBAC drop rate:                                        56%
  Parent-child reconstructions performed:                0

  Scenarios where access was denied: 0 / 5

Exported: D:\URV\TFG\ai-rag-context-auth-system\strategy-analysis\notebooks\ph3-multiagent-routing-strategy\..\..\data\results\notebook_results\ph3\ph3_step4_hierarchical_agent.json
Exported: D:\URV\TFG\ai-rag-context-auth-system\strategy-analysis\noteboo